# Similarity and Record Linkage

## Objective
Compare Bloom filters and identify matches.

In [4]:
import pandas as pd

df_A = pd.read_pickle("../data/processed/df_A_blocked.pkl")
df_B = pd.read_pickle("../data/processed/df_B_blocked.pkl")

### Dice similarity

In [5]:
def dice_similarity(bf1, bf2):
    intersection = (bf1 & bf2).count()
    return 2 * intersection / (bf1.count() + bf2.count() + 1e-10)

### Generate pairs

In [6]:
def generate_pairs(df_A, df_B, block):
    pairs = []
    for val in df_A[block].dropna().unique():
        subA = df_A[df_A[block] == val]
        subB = df_B[df_B[block] == val]

        for i in subA.index:
            for j in subB.index:
                pairs.append((i, j))
    return pairs

pairs = set()
for b in ["block1", "block2", "block3"]:
    pairs.update(generate_pairs(df_A, df_B, b))

matches = []
for i, j in pairs:
    sim = dice_similarity(df_A.loc[i,"bloom"], df_B.loc[j,"bloom"])
    if sim >= 0.85:
        matches.append({
            "id_A": df_A.loc[i,"id"],
            "id_B": df_B.loc[j,"id"],
            "sim": sim
        })

Candidate pairs → similarity → threshold
<br>Produces predicted matches

In [7]:
matches_df = pd.DataFrame(matches)
matches_df.to_csv("../data/processed/matches.csv", index=False)